# 17 · Hybrid Search：稠密 + 稀疏 + 分数融合

> Dense 与 Sparse 分数**量纲不同，不能直接相加**。融合策略是混合检索的灵魂。

**本文件覆盖知识点**：Dense+Sparse / Score Fusion / Weighted Fusion / Reciprocal Rank Fusion(RRF) / Weighted RRF / Query Weight / Relative Score Fusion

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 三类融合策略

| 策略 | 做法 | 特点 |
|------|------|------|
| **Score Fusion** | 把两边分数映射到同一量纲后相加 | 保留强度，需归一化（min-max/z-score） |
| **Weighted Fusion** | 加权求和：`w1·Sparse + w2·Dense` | 可调权重；仍要解决量纲 |
| **RRF** | 只看名次：`Σ 1/(k+rank)` | **无需统一量纲**，稳健，最常用 |

RRF 公式：`RRF(doc) = Σ over lists  1 / (k + rank_i(doc))`，`k` 常取 60。

> RRF 的精妙：它不在乎“BM25 给 12 分、向量给 0.83”这种不可比的分数，只看**各自榜单里的名次**。

In [ ]:
# 两个榜单用 RRF 融合的微型演示
def rrf_fuse(rankings, k=60):
    """rankings: 多个"文档id榜单"(从高到低)。返回融合分降序的 [(doc, score)]"""
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] = score.get(doc, 0) + 1.0 / (k + rank + 1)
    return sorted(score.items(), key=lambda x: -x[1])

vec_list  = ['A', 'C', 'B', 'E']   # 向量检索榜
bm25_list = ['B', 'D', 'A', 'C']   # BM25 榜

print('融合结果:', [(d, round(s, 4)) for d, s in rrf_fuse([vec_list, bm25_list])])
print('=> 两边都靠前的 A、B 排最前；只被一方召回的 D/E 次之。')

## 2. 何时升级融合策略

- 起步直接用 **RRF**，几乎不用调参；
- 想要更优：**Weighted RRF / Relative Score Fusion**——给 Sparse、Dense 不同权重，或用每个列表内部相对的分数形态（如归一化到 [0,1]）加权；
- **Query Weight**：有些问题明显是“查精确型号”→ 临时调高 Sparse 权重。

融合后仍要再走 **Rerank（第22课）** 做最终精排，两段职责不同。



In [ ]:
# 知识点·真调说明：Query Weight —— 让模型把查询判成「精确词面 / 语义概念」，决定融合时该偏向 Sparse 还是 Dense
import json as _json
_fall = ('[{"query": "星云客服机器人 企业版 型号 SE-2200 的参数", "type": "exact", "reason": "型号是唯一词面标识"}, '
         '{"query": "客服机器人私有化部署需要满足哪些合规要求？", "type": "semantic", "reason": "合规要求需概念召回"}, '
         '{"query": "报错码 KB-503 是什么意思", "type": "exact", "reason": "报错码靠精确匹配"}, '
         '{"query": "客服机器人与人工客服相比，优势是什么？", "type": "semantic", "reason": "对比类需语义泛化"}]')
out = _llm_live(
    prompt="""判断下面 4 条用户查询在混合检索里更适合“关键词精确匹配（偏向 BM25 / Sparse）”还是“语义理解（偏向向量 / Dense）”，输出 JSON 数组，不要输出任何其它文字。

1. 星云客服机器人 企业版 型号 SE-2200 的参数
2. 客服机器人私有化部署需要满足哪些合规要求？
3. 报错码 KB-503 是什么意思
4. 客服机器人与人工客服相比，优势是什么？

JSON 格式：[{"query": 查询原文, "type": "exact" 或 "semantic", "reason": 一句话理由}]""",
    system='你是混合检索的查询分析器。exact=词面命中即有答案（型号/报错码/精确名称）；semantic=需要同义或概念泛化（比较/合规/优劣）。',
    fallback=_fall,
    temperature=0.2,
)
s = out if out is not None else _fall
s = s.strip().strip('`')
if s.startswith('json'):
    s = s[4:].strip()
if out is None:
    print('（以上为固定样例；下面用样例走 json.loads 解析）')
try:
    for it in _json.loads(s):
        if it['type'] == 'exact':
            tag = '→ 词面即答案：融合时该给 Sparse(BM25) 更大权重'
        else:
            tag = '→ 语义泛化：融合时该给 Dense(向量) 更大权重'
        print('• %s（%s）%s' % (it['query'], it['reason'], tag))
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明结构化约束需再收紧')
print('→ 这就是 Query Weight 的输入：先判定查询类型，再临时调高 Sparse 或 Dense 的权重；RRF 本身无权重，可用 Weighted RRF 落地。')

## 3. 一个生产级检索的完整形态（预告）

```text
BM25 榜 ─┐
Dense 榜 ─┼─→ RRF 融合 → Top-50 候选池 → Rerank → Top-5 → LLM
Metadata ─┘   (前置过滤)
```

## 小结

- Hybrid = Dense + Sparse，量纲不同需**融合**；
- **RRF 按名次融合**，稳健零调参，是标配；追求更好再上 Weighted / 分数归一；
- 混合检索之后通常接 Rerank。